# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/vandanara/UofUtah_IS4487/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [58]:
df = pd.read_csv('cleaned_airbnb_data.csv')
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20250906174623,2025-09-06,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://a0.muscache.com/pictures/44627226/0e72...,4621559,...,4.22,4.56,3.22,3.67,f,1,1,0,0,0.07
1,3820211,https://www.airbnb.com/rooms/3820211,20250906174623,2025-09-06,city scrape,Funky Urban Gem: Prime Central Location - Park...,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.85,4.81,4.81,4.77,f,5,5,0,0,2.31
2,5651579,https://www.airbnb.com/rooms/5651579,20250906174623,2025-09-06,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",The neighborhood is very eclectic. We have a v...,https://a0.muscache.com/pictures/b3fc42f3-6e5e...,29288920,...,4.81,4.88,4.76,4.64,f,2,1,1,0,2.96
3,6623339,https://www.airbnb.com/rooms/6623339,20250906174623,2025-09-06,city scrape,Cozy City Stay · Free Parking + Walkable Location,Step into the charming and comfy 1BR/1BA apart...,Overview<br /><br />The lovely apartment is lo...,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.83,4.70,4.80,4.72,f,5,5,0,0,2.66
4,9005989,https://www.airbnb.com/rooms/9005989,20250906174623,2025-09-06,city scrape,"Studio in The heart of Center SQ, in Albany NY",(21 years of age or older ONLY) NON- SMOKING.....,"There are many shops, restaurants, bars, museu...",https://a0.muscache.com/pictures/d242a77e-437c...,17766924,...,4.95,4.93,4.87,4.77,f,1,1,0,0,5.63


### ✍️ Your Response: 🔧
1. This dataset contains a comprehensive collection of Airbnb listing information, covering 76 different columns. It includes core listing details such as the listing ID, URL, name, description, and main photo, along with the scrape date showing when the data was collected. It also provides host-related information, including the host’s ID and the number of listings they manage across different room types. The dataset includes location context through fields like neighborhood overview, giving insight into the surrounding area. Additionally, it contains a full set of review and rating metrics—for cleanliness, communication, check-in, value, location, and overall guest satisfaction—as well as booking-related fields like whether the listing is instant bookable and how many reviews it receives per month. Overall, the dataset offers a detailed view of both listing attributes and host performance, making it well-suited for analysis of pricing, quality, demand, and guest experience.

2. 465 rows are presented and 76 columns

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [59]:
# Drop columns that do not help predict price
cols_to_drop = [
    'id',
    'listing_url',
    'scrape_id',
    'last_scraped',
    'source',
    'name',
    'description',
    'neighborhood_overview',
    'picture_url',
    'host_id'
]

df_clean = df.drop(columns=cols_to_drop, errors='ignore')
df_clean.head()

,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,https://www.airbnb.com/users/show/4621559,Kenneth,2013-01-07,"New York, NY",I am a real down to earth & cool person.,NaN,Unknown,50%,f,https://a0.muscache.com/im/users/4621559/profi...,...,4.22,4.56,3.22,3.67,f,1,1,0,0,0.07
1,https://www.airbnb.com/users/show/19648678,Terra,2014-08-07,"Albany, NY","Hello! I’m a proud resident of Albany, NY, whe...",within an hour,100%,100%,t,https://a0.muscache.com/im/pictures/user/User/...,...,4.85,4.81,4.81,4.77,f,5,5,0,0,2.31
2,https://www.airbnb.com/users/show/29288920,Gregg,2015-03-13,"Albany, NY",I am an Albany native .I have lived in Ireland...,within an hour,100%,99%,f,https://a0.muscache.com/im/users/29288920/prof...,...,4.81,4.88,4.76,4.64,f,2,1,1,0,2.96
3,https://www.airbnb.com/users/show/19648678,Terra,2014-08-07,"Albany, NY","Hello! I’m a proud resident of Albany, NY, whe...",within an hour,100%,100%,t,https://a0.muscache.com/im/pictures/user/User/...,...,4.83,4.70,4.80,4.72,f,5,5,0,0,2.66
4,https://www.airbnb.com/users/show/17766924,Sugey,2014-07-07,"Albany, NY",NaN,NaN,Unknown,100%,t,https://a0.muscache.com/im/pictures/user/ae311...,...,4.95,4.93,4.87,4.77,f,1,1,0,0,5.63


### ✍️ Your Response: 🔧
1. the columns that I dropper were Id, listing_url, scrape_id, last_scraped, source, name, description, neightborhood_overview, picture_url and host_id because these they do not provide meaningful or usable information for predicting price. Fields like ID numbers, URLs, picture links, and scrape timestamps are simply identifiers with no relationship to listing value, and including them would introduce noise. Text-heavy fields such as name, description, and neighborhood_overview are unstructured and require complex NLP preprocessing to be useful, which is outside the scope of a standard regression model. Removing these features keeps the dataset focused on features that actaully influence price.

2. Including these columns in the model introduces several risks. Identifier fields like IDs and URLs can mislead the model into learning meaningless numeric patterns, causing overfitting and reducing generalizability. Text-heavy columns, such as descriptions or neighborhood overview,s can also cause errors because regression models cannot process raw text without preprocessing, leading to failed training or distorted predictions. These fields add noise rather than signal, making the model less accurate, harder to interpret, and more prone to instability.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [61]:
numeric_df = df_clean.select_dtypes(include=['number'])
correlation_matrix = numeric_df.corr()
price_correlations = correlation_matrix['price'].sort_values(ascending=False)
print("Correlation with 'price':")
display(price_correlations)

Correlation with 'price':


,price
price,1.000000
accommodates,0.630254
bedrooms,0.578843
beds,0.554078
bathrooms,0.442306
estimated_revenue_l365d,0.271374
maximum_maximum_nights,0.148297
maximum_nights_avg_ntm,0.140179
minimum_maximum_nights,0.133550
calculated_host_listings_count_private_rooms,0.045514


### ✍️ Your Response: 🔧
1. The variables with the strongest positive correlation to price were accommodates, bedrooms, beds, and bathrooms, with accommodates showing the highest correlation at 0.63. These features reflect the size/capacity of the listing and naturally drive higher pricing

2. Based on the results, accommodations, bedrooms, beds, bathrooms, and estimated_revenue_l365d would be strong candidates because they capture property capacity and earning potential. Some of the moderate correlations, like maximum_nights_avg_ntm and maximum_maximum_nights, may also add predictive value when combined with other features. Meanwhile, weak or near-zero correlations (like most review scores and host listing counts) would contribute little to a pricing model.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [63]:
features = ['accommodates', 'bedrooms', 'beds', 'bathrooms']
target = 'price'
x = df_clean[features]
y = df_clean[target]

### ✍️ Your Response: 🔧
1. the featurea that I am using are accomodates, bedrooms, beds, and bathrooms

2. This is a regression problem because we are predicting a numeric value, the nightly price of an Airbnb listing. Prices can take on a wide range of continuous values ($75, $120, $143.50, etc.), and the objective is to estimate that number as accurately as possible. Classification models only work when the output falls into distinct categories (like “low / medium / high” or “yes/no”). Since price is continuous, regression is the correct approach.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [64]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (372, 39)
X_test shape: (93, 39)
y_train shape: (372,)
y_test shape: (93,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [73]:
# Initialize the Linear Regression model
model = LinearRegression()

# Combine X_train and y_train to drop rows with NaNs consistently
train_df = pd.concat([X_train, y_train], axis=1)
train_df_cleaned = train_df.dropna()

X_train_cleaned = train_df_cleaned.drop(columns=['price'])
y_train_cleaned = train_df_cleaned['price']

# Make sure X_test and y_test are also clean, though the error was in training
# For simplicity, we'll drop NaNs from test sets as well if they exist
test_df = pd.concat([X_test, y_test], axis=1)
test_df_cleaned = test_df.dropna()

X_test_cleaned = test_df_cleaned.drop(columns=['price'])
y_test_cleaned = test_df_cleaned['price']


# Fit the model to the cleaned training data
model.fit(X_train_cleaned, y_train_cleaned)

# Make predictions on the cleaned test set
y_pred = model.predict(X_test_cleaned)

print("Linear Regression model fitted and predictions made.")

Linear Regression model fitted and predictions made.


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [67]:
# Calculate Mean Squared Error (MSE)
mse = mean_squared_error(y_test_cleaned, y_pred)

# Calculate R-squared (R²)
r2 = r2_score(y_test_cleaned, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

Mean Squared Error (MSE): 5335.78
R-squared (R²): 0.45


### ✍️ Your Response: 🔧
1. The R² score is 0.45, which means the model explains about 45% of the price variation. That’s a moderate fit; the model captures some meaningful patterns, but a large portion of the price variation is still unexplained.

2. The MSE is 5335.78, which is relatively high for predicting Airbnb prices, indicating that the model’s predictions are off by a substantial amount on average. To improve it, you could add more predictive features, engineer new variables, try non-linear models, or apply regularization techniques to capture more complex relationships.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [76]:
coefficients_df = pd.DataFrame({
    'Feature': X_train_cleaned.columns,
    'Coefficient': model.coef_
})

coefficients_df['Abs'] = coefficients_df['Coefficient'].abs()
coefficients_df = coefficients_df.sort_values(by='Abs', ascending=False)

display(coefficients_df)

,Feature,Coefficient,Abs
2,latitude,-1143.704319,1143.704319
3,longitude,-473.072359,473.072359
33,review_scores_value,71.301269,71.301269
30,review_scores_checkin,-55.064894,55.064894
31,review_scores_communication,-47.927385,47.927385
27,review_scores_rating,-28.662532,28.662532
4,accommodates,27.462059,27.462059
29,review_scores_cleanliness,26.783521,26.783521
28,review_scores_accuracy,22.720989,22.720989
6,bedrooms,-18.499451,18.499451


### ✍️ Your Response: 🔧
1. Latitude and longitude had the strongest positive impact in magnitude, meaning location, especially where a listing sits on the map, drives large pricing differences. Among the interior features, review-related scores such as value, cleanliness, and accuracy also showed positive coefficients, indicating that higher guest ratings tend to push prices upward. Accommodates also had a meaningful positive effect, reflecting that larger listings can charge more.

2. Several review score categories, including check-in and communication, displayed negative coefficients. This result is counterintuitive because we would usually expect higher reviews to correlate with increased prices. Similarly, bedrooms had a negative coefficient, unexpectedly so, since more bedrooms generally lead to higher pricing. These trends may indicate multicollinearity or overlapping effects between features related to size and those associated with location attributes.

3. Location appears to be the dominant driver of price, far outweighing individual listing features. Review quality also meaningfully affects pricing, reinforcing the importance of maintaining high guest satisfaction. The unexpected negative coefficients highlight that some features overlap, suggesting hosts should focus on overall value and guest experience rather than simply adding more rooms or adjusting minimum stays.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [77]:
# 1. Choose the top 5 features based on strongest absolute coefficients
refined_features = [
    'latitude',
    'longitude',
    'review_scores_value',
    'review_scores_checkin',
    'review_scores_communication'
]

# Prepare data for the refined model
X_refined = numeric_df[refined_features]
y_refined = numeric_df['price']

# Re-split data for the refined model
X_train_refined, X_test_refined, y_train_refined, y_test_refined = train_test_split(X_refined, y_refined, test_size=0.2, random_state=42)

# Handle NaNs for the refined model's training and testing sets
train_df_refined = pd.concat([X_train_refined, y_train_refined], axis=1)
train_df_refined_cleaned = train_df_refined.dropna()

X_train_refined_cleaned = train_df_refined_cleaned.drop(columns=['price'])
y_train_refined_cleaned = train_df_refined_cleaned['price']

test_df_refined = pd.concat([X_test_refined, y_test_refined], axis=1)
test_df_refined_cleaned = test_df_refined.dropna()

X_test_refined_cleaned = test_df_refined_cleaned.drop(columns=['price'])
y_test_refined_cleaned = test_df_refined_cleaned['price']

# 2. Rebuild the regression model using just those features
refined_model = LinearRegression()
refined_model.fit(X_train_refined_cleaned, y_train_refined_cleaned)

# Make predictions on the refined test set
y_pred_refined = refined_model.predict(X_test_refined_cleaned)

# 3. Compare MSE and R² between the baseline and refined model
mse_refined = mean_squared_error(y_test_refined_cleaned, y_pred_refined)
r2_refined = r2_score(y_test_refined_cleaned, y_pred_refined)

print(f"Baseline Model MSE: {mse:.2f}")
print(f"Baseline Model R-squared: {r2:.2f}")
print("----------------------------------")
print(f"Refined Model MSE: {mse_refined:.2f}")
print(f"Refined Model R-squared: {r2_refined:.2f}")

Baseline Model MSE: 5335.78
Baseline Model R-squared: 0.45
----------------------------------
Refined Model MSE: 9678.61
Refined Model R-squared: -0.00


### ✍️ Your Response: 🔧
1. I kept the five features with the strongest absolute coefficients: latitude, longitude, review_scores_value, review_scores_checkin, and review_scores_communication. These were selected because they showed the greatest overall impact on price in the baseline model, making them logical candidates for testing a simplified, higher-signal model.

2. No, model performance worsened significantly. The R² dropped from 0.45 to 0.00, and the MSE nearly doubled. This happened because the refined model removed too many predictive features, losing important information related to room size, capacity, and demand patterns that matter for pricing. The top coefficients alone did not capture enough price variation to sustain performance.

3. I would recommend the baseline model. It explains far more price variation (R² = 0.45) and produces much lower error. While the refined model is more compact, it loses predictive accuracy, making it unsuitable for reliable pricing recommendations

4. This connects to my learning outcomes because evaluating both models strengthens my ability to interpret analytical results and judge which variables actually drive outcomes, skills that align with market risk analysis and investment evaluation. It also reinforces communicating data insights clearly to stakeholders, which is part of my goal to develop strong professional communication for finance and consulting roles.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. The model clarified which listing features have the greatest influence on price, helping identify what drives revenue and how hosts can adjust their offerings to stay competitive.

2. I’d recommend prioritizing improvements in the features most strongly tied to price, such as location-related elements and overall listing quality scores. Enhancing accuracy, cleanliness, and communication ratings can help hosts justify higher pricing.

3. Next steps would include adding more relevant features, incorporating categorical variables through encoding, and testing more advanced models to capture non-linear trends. This would increase predictive accuracy and generate insights that are more actionable for pricing strategy.

4. This connects to my learning outcomes because evaluating both models strengthens my ability to interpret analytical results and judge which variables actually drive outcomes, skills that align with market risk analysis and investment evaluation. It also reinforces communicating data insights clearly to stakeholders, which is part of my goal to develop strong professional communication for finance and consulting roles.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [78]:
!jupyter nbconvert --to html "assignment_11_LastnameFirstname.ipynb"

[NbConvertApp] Converting notebook assignment_11_LastnameFirstname.ipynb to html
[NbConvertApp] Writing 364902 bytes to assignment_11_LastnameFirstname.html


# Task
Rebuild the regression model using 'latitude', 'longitude', 'review_scores_value', 'review_scores_checkin', and 'review_scores_communication' as features, then evaluate its performance using Mean Squared Error (MSE) and R-squared (R²), comparing the results to the baseline model.

## rebuild_and_evaluate_refined_model

### Subtask:
Rebuild the regression model using a selected subset of features ('latitude', 'longitude', 'review_scores_value', 'review_scores_checkin', 'review_scores_communication'), then evaluate its performance using MSE and R², and compare it to the baseline model.


## Summary:

### Data Analysis Key Findings
-   The objective of the current step is to rebuild a regression model using a specific set of features: 'latitude', 'longitude', 'review_scores_value', 'review_scores_checkin', and 'review_scores_communication'.
-   The performance of this refined model will be evaluated using two key metrics: Mean Squared Error (MSE) and R-squared (R\^2).
-   A direct comparison of the refined model's performance against a previously established baseline model is planned to assess its effectiveness and improvement.

### Insights or Next Steps
-   The upcoming evaluation will reveal whether the selected feature subset significantly improves model accuracy and explanatory power compared to the baseline, as indicated by lower MSE and higher R\^2 values.
-   Depending on the results, further iterations of feature selection or engineering may be considered if the refined model does not provide substantial gains over the baseline.
